# 12.1 · RNN, LSTM, GRU / 循环神经网络

> **课程定位 / Where this fits**
> 第 1 课，**Part 12 · 现代 NLP 与大语言模型**。通往 Transformer 的起点。
> Lesson 1, **Part 12 · Modern NLP & LLMs**. The starting point on the road to Transformers.
>
> 文本、语音、时间序列都是**序列**——长度可变、前后有依赖。前馈网络/CNN 不擅长处理"任意长且有顺序记忆"的数据。**循环神经网络(RNN)** 通过一个**随时间传递的隐藏状态**来"记住"历史。但朴素 RNN 有致命的**梯度消失**问题，记不住长距离依赖。**LSTM/GRU** 用**门控**机制解决它，统治了 2014–2017 年的 NLP，也是理解 Transformer 注意力"为何出现"的关键背景。本课**从零实现 RNN**、**亲眼看到梯度消失**、并验证 **LSTM 在长依赖上完胜 RNN**。
> Text, speech, time series are **sequences** — variable length, with dependencies. Feedforward/CNN aren't built for "arbitrarily long, order-dependent" data. **RNNs** "remember" history via a **hidden state passed through time**. But vanilla RNNs suffer from **vanishing gradients** and can't capture long-range dependencies. **LSTM/GRU** fix this with **gates**, dominating NLP in 2014–2017 and providing key context for why attention/Transformers arose. We **implement an RNN from scratch**, **witness vanishing gradients**, and verify **LSTM beats RNN on long dependencies**.
>
> 💼 **实战/面试视角**："RNN 怎么处理序列 / 梯度消失为什么 / LSTM 门控如何解决 / LSTM vs GRU" 是序列建模必考。
> 💼 **Practical/interview angle:** "how RNNs handle sequences / why vanishing gradients / how LSTM gates fix it / LSTM vs GRU" — sequence-modeling essentials.

> 📐 **符号约定 / Notation**
> - $h_t$ —— 第 $t$ 步的隐藏状态(记忆) / hidden state (memory) at step $t$
> - $W_{xh}, W_{hh}$ —— 输入→隐藏、隐藏→隐藏 的权重 / input→hidden, hidden→hidden weights
> - 门(gate) —— 0~1 的开关, 控制信息流 / a 0–1 valve controlling information flow

> 💡 **面试相关 / Interview-relevant**
> - "RNN 的循环结构/隐藏状态"（出镜率 ★★★★）
> - "梯度消失/爆炸为什么发生"（★★★★★）
> - "LSTM 的三个门和 cell state"（★★★★★）
> - "LSTM vs GRU 的区别"（★★★★）
> - "RNN 为何被 Transformer 取代"（★★★★，无法并行+长依赖）

---

## 学习目标 / Learning Objectives
1. 理解循环与隐藏状态如何处理变长序列。
   Understand recurrence and hidden state for variable-length sequences.
2. **从零实现 RNN** 前向传播。
   Implement RNN forward from scratch.
3. **可视化梯度消失**——朴素 RNN 的根本缺陷。
   Visualize vanishing gradients — the vanilla RNN's core flaw.
4. 理解 **LSTM/GRU 门控**如何解决长依赖。
   Understand how LSTM/GRU gates solve long dependencies.
5. 实验验证 **LSTM 在长依赖上胜过 RNN**。
   Experimentally verify LSTM beats RNN on long dependencies.

## 目录 / TOC
1. [循环与隐藏状态（从零）⭐](#1)
2. [梯度消失：朴素 RNN 的硬伤 ⭐](#2)
3. [LSTM 与 GRU：门控机制 ⭐](#3)
4. [实验：长依赖任务 RNN vs LSTM ⭐](#4)


<a id="1"></a>
## 1. 循环与隐藏状态（从零）⭐ / Recurrence & Hidden State

前馈网络一次吃一个**固定大小**的输入。但句子有长有短、且**词的顺序和上下文很重要**。RNN 的想法：**一次处理一个时间步**，并维护一个**隐藏状态 $h_t$**(可看作"到目前为止的记忆摘要")，每来一个新输入就更新它：
A feedforward net takes one **fixed-size** input. But sentences vary in length and **order/context matter**. RNN's idea: **process one time step at a time**, maintaining a **hidden state $h_t$** (a "memory summary so far"), updated with each new input:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)$$

关键：**同一组权重 $W$ 在每个时间步重复使用**(参数共享，像 CNN 的核在空间上共享，RNN 在时间上共享)，所以能处理任意长度的序列。$h_{t-1}$ 把过去的信息带到现在——这就是"记忆"。
Key: **the same weights $W$ are reused at every time step** (parameter sharing — like CNN kernels in space, RNN shares across time), so it handles any length. $h_{t-1}$ carries the past into the present — that's "memory."

下面**从零实现** RNN 前向，看隐藏状态如何随序列演变。
Let's implement RNN forward **from scratch** and watch the hidden state evolve.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

def rnn_forward(xs, Wxh, Whh, b, h0):
    """从零的 RNN 前向: 逐时间步更新隐藏状态 / RNN forward from scratch, step by step."""
    h = h0; H = []
    for x in xs:                                          # 遍历每个时间步 / iterate over time steps
        h = np.tanh(Wxh @ x + Whh @ h + b)                # 隐藏状态更新公式 / hidden-state update
        H.append(h)
    return np.array(H)

D_in, D_h, T = 3, 4, 8                                     # 输入维, 隐藏维, 序列长度 / dims & length
Wxh = np.random.randn(D_h, D_in)*0.5; Whh = np.random.randn(D_h, D_h)*0.5; b = np.zeros(D_h)
xs = np.random.randn(T, D_in)                             # 一条长度为8的序列 / a length-8 sequence
H = rnn_forward(xs, Wxh, Whh, b, np.zeros(D_h))
print(f"输入序列形状 {xs.shape} (T={T}个时间步, 每步{D_in}维)")
print(f"隐藏状态序列形状 {H.shape} (每步一个{D_h}维隐藏状态)")
fig, ax = plt.subplots(figsize=(8, 3.2))
sns.heatmap(H.T, cmap="coolwarm", center=0, cbar_kws={"label":"隐藏单元值"},
            xticklabels=[f"t{t}" for t in range(T)], yticklabels=[f"h{i}" for i in range(D_h)], ax=ax)
ax.set_xlabel("时间步"); ax.set_title("RNN 隐藏状态随时间演变(同一组权重在每步复用)")
plt.tight_layout(); plt.show()
print("每个时间步用相同权重更新隐藏状态; h_t 携带历史信息 → '记忆'; 参数量与序列长度无关")


<a id="2"></a>
## 2. 梯度消失：朴素 RNN 的硬伤 ⭐ / Vanishing Gradients

RNN 训练靠**沿时间反向传播(BPTT)**：把循环"展开"成一个很深的网络(深度=序列长度)再反传。问题来了——梯度要穿过很多个时间步，每步都乘上 $W_{hh}$ 和 $\tanh'$(都通常 <1)。**连乘很多个小于 1 的数 → 指数级趋近 0**：早期时间步几乎收不到梯度，**学不到长距离依赖**。这就是**梯度消失(vanishing gradient)**(呼应 9.9、10.4 的深度网络梯度问题)。
RNN training uses **backprop through time (BPTT)**: unroll the recurrence into a very deep net (depth = sequence length) and backprop. The catch — gradients must pass through many steps, each multiplying by $W_{hh}$ and $\tanh'$ (typically <1). **Multiplying many sub-1 numbers → exponential decay toward 0**: early steps get almost no gradient and **can't learn long-range dependencies**. This is the **vanishing gradient** (echoing 9.9, 10.4).

下面直接测量：把序列末端输出的梯度反传到**每个输入时间步**，看梯度大小如何随"距离末端越远"而衰减。对比朴素 RNN 和 LSTM。
Let's measure directly: backprop the gradient of the final output to **each input time step**, and see how it decays with distance from the end. Compare vanilla RNN vs LSTM.


In [ ]:
def grad_over_time(kind, T=60, D=16):
    """测量末端输出对各输入时间步的梯度范数 / gradient norm of final output w.r.t. each input step."""
    torch.manual_seed(0)
    rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[kind](D, D, batch_first=True)
    x = torch.randn(1, T, D, requires_grad=True)          # 输入序列(要求梯度) / input requiring grad
    out, _ = rnn(x)
    out[:, -1].norm().backward()                          # 从最后一步输出反传 / backprop from last step
    g = x.grad[0].norm(dim=1).numpy()                     # 每个时间步的输入梯度范数 / per-step grad norm
    return g

g_rnn = grad_over_time("RNN"); g_lstm = grad_over_time("LSTM")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(g_rnn, "s-", color="#e67", label="朴素 RNN (越往前梯度越小→消失)")
ax.plot(g_lstm, "o-", color="#39c", label="LSTM (梯度更平稳)")
ax.set_yscale("log"); ax.set_xlabel("输入时间步 (0=最早 → 最后)"); ax.set_ylabel("梯度范数(对数轴)")
ax.legend(); ax.set_title("梯度消失: 末端输出对'早期时间步'的梯度, 朴素RNN指数衰减")
plt.tight_layout(); plt.show()
print(f"朴素RNN: 最早时间步梯度 {g_rnn[0]:.2e} vs 最后 {g_rnn[-1]:.2e}  (相差很多数量级→早期学不动)")
print(f"LSTM:   最早时间步梯度 {g_lstm[0]:.2e} vs 最后 {g_lstm[-1]:.2e}  (更平稳→能记住更远)")
print("根因: BPTT 让梯度连乘很多 <1 的项 → 指数消失; 早期步收不到信号 → 学不到长依赖")


<a id="3"></a>
## 3. LSTM 与 GRU：门控机制 ⭐ / LSTM & GRU: Gating

**LSTM(长短期记忆网络)** 的核心创新是引入一条**细胞状态(cell state) $c_t$**——一条贯穿时间的"**信息高速公路**"，信息可以**几乎无损地直线流过**(类比 10.4 ResNet 的跳跃连接！)。用三个**门(gate)**(0~1 的阀门，由 sigmoid 算出)来控制读写：
The core innovation of **LSTM (Long Short-Term Memory)** is a **cell state $c_t$** — an "**information highway**" through time that info can **flow along almost unchanged** (like ResNet's skip connection, 10.4!). Three **gates** (0–1 valves from sigmoid) control reads/writes:
- **遗忘门(forget gate)**：决定从 cell state **丢弃**多少旧信息。
  **Forget gate:** how much old info to **drop** from the cell state.
- **输入门(input gate)**：决定**写入**多少新信息。
  **Input gate:** how much new info to **write**.
- **输出门(output gate)**：决定从 cell state **读出**多少作为本步隐藏状态。
  **Output gate:** how much to **read out** as this step's hidden state.

**为什么解决梯度消失**(面试核心)：当遗忘门≈1、输入门≈0 时，$c_t \approx c_{t-1}$——cell state 沿时间**加性**传递(而非朴素 RNN 的反复**乘性**变换)，梯度可以沿这条"高速公路"基本不衰减地回传。**加性路径 = 梯度高速公路**，和 ResNet 的 $+x$ 异曲同工。
**Why it fixes vanishing gradients** (interview core): when forget≈1, input≈0, $c_t \approx c_{t-1}$ — the cell state passes through **additively** (not the repeated **multiplicative** transforms of vanilla RNN), so gradients flow back along this "highway" almost undamped. **Additive path = gradient highway**, just like ResNet's $+x$.

**GRU(门控循环单元)** 是 LSTM 的简化：合并成**两个门**(更新门 + 重置门)、去掉单独的 cell state。**参数更少、更快**，效果常和 LSTM 相当——这是常考的对比。
**GRU (Gated Recurrent Unit)** simplifies LSTM into **two gates** (update + reset), dropping the separate cell state. **Fewer parameters, faster**, often comparable to LSTM — a common interview comparison.


In [ ]:
# 对比三者的参数量(同样隐藏维度) / parameter counts (same hidden size)
D = 64
for kind, cls in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    m = cls(D, D, batch_first=True)
    n = sum(p.numel() for p in m.parameters())
    print(f"{kind:5}: {n:,} 参数  ({'1组权重' if kind=='RNN' else '2个门' if kind=='GRU' else '3个门+cell'})")
print("\nRNN<GRU<LSTM(门越多参数越多); GRU 用更少参数达到接近 LSTM 的效果, 训练更快")
print("LSTM cell state 的加性传递 = 梯度高速公路(类比 ResNet 的 +x), 这是它能记长依赖的关键")


<a id="4"></a>
## 4. 实验：长依赖任务 RNN vs LSTM ⭐ / Long-Dependency Experiment

用经典的 **"加法问题"(adding problem)** —— LSTM 论文里用来证明长依赖能力的基准。输入一个长度 $T$ 的序列，每个时间步有两个特征：一个 $[0,1]$ 随机数 + 一个**标记位**(整条序列里恰好 2 个位置标记为 1)。目标：**输出这两个被标记位置上数字的和**。
We use the classic **adding problem** — the LSTM-paper benchmark for long dependencies. Input: a length-$T$ sequence; each step has two features: a random number in $[0,1]$ + a **marker** (exactly 2 positions are marked 1). Target: **the sum of the two marked numbers**.

这要求模型**精确记住**早期被标记的数字，跨越很多无关步——正是考验长依赖。我们对比短序列(T=10)和长序列(T=40)下 RNN 和 LSTM 的表现。
This requires **precisely remembering** an early marked number across many irrelevant steps — a long-dependency test. We compare RNN vs LSTM at short (T=10) and long (T=40) sequences.


In [ ]:
def adding_problem(T, n=4000, seed=0):
    rng = np.random.RandomState(seed)
    vals = torch.rand(n, T, 1)                             # 随机数特征 / random values
    mark = torch.zeros(n, T, 1); y = torch.zeros(n, 1)
    for i in range(n):
        a, b = rng.choice(T, 2, replace=False)             # 随机选2个位置标记 / mark 2 positions
        mark[i, a] = 1; mark[i, b] = 1; y[i] = vals[i, a] + vals[i, b]   # 目标=两标记数之和 / target = sum
    return torch.cat([vals, mark], dim=2), y               # 特征=(值,标记) / features = (value, marker)

class SeqRegressor(nn.Module):
    def __init__(self, kind, h=48):
        super().__init__()
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[kind](2, h, batch_first=True)
        if kind == "LSTM":                                 # 把遗忘门 bias 初始化为1(标准技巧, 初始倾向"记住") / forget-bias=1
            for name, p in self.rnn.named_parameters():
                if "bias" in name:
                    nn.init.zeros_(p); hh = p.shape[0]; p.data[hh//4:hh//2] = 1.0
        self.fc = nn.Linear(h, 1)
    def forward(self, x): out, _ = self.rnn(x); return self.fc(out[:, -1])

def train_adding(kind, T, epochs=15):
    Xtr, ytr = adding_problem(T, seed=0); Xte, yte = adding_problem(T, n=800, seed=1)
    torch.manual_seed(0); m = SeqRegressor(kind); opt = torch.optim.Adam(m.parameters(), 3e-3); mse = nn.MSELoss()
    for _ in range(epochs):
        for i in range(0, len(Xtr), 128):
            opt.zero_grad(); mse(m(Xtr[i:i+128]), ytr[i:i+128]).backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()      # 梯度裁剪防爆炸(见9.11) / grad clip
    return mse(m(Xte), yte).item()

results = {}
for T in [10, 40]:
    for kind in ["RNN", "LSTM"]:
        results[(kind, T)] = train_adding(kind, T)
        print(f"T={T:2}  {kind:4}: 测试 MSE = {results[(kind,T)]:.3f}")
baseline = 0.167                                           # 瞎猜(预测均值)的 MSE≈方差 / chance MSE ≈ variance
fig, ax = plt.subplots(figsize=(6.5,4))
x = np.arange(2); w = 0.35
ax.bar(x-w/2, [results[("RNN",10)], results[("RNN",40)]], w, label="RNN", color="#e67")
ax.bar(x+w/2, [results[("LSTM",10)], results[("LSTM",40)]], w, label="LSTM", color="#39c")
ax.axhline(baseline, color="gray", ls="--", label="瞎猜基线(≈方差)")
ax.set_xticks(x); ax.set_xticklabels(["T=10 (短)","T=40 (长)"]); ax.set_ylabel("测试 MSE(越低越好)"); ax.legend()
ax.set_title("加法问题: 长序列(T=40)时朴素RNN退化到瞎猜, LSTM 仍能学")
plt.tight_layout(); plt.show()
print(f"\nT=10: RNN/LSTM 都能学; T=40: RNN MSE≈{results[('RNN',40)]:.2f}≈瞎猜({baseline}) 完全失败, LSTM 仍工作")
print("结论: 长依赖下朴素RNN因梯度消失失败; LSTM 靠门控+cell state 高速公路记住远处信息")


```
RNN: 逐时间步更新隐藏状态 h_t=tanh(Wxh·x+Whh·h+b); 同一组权重每步复用(时间上参数共享)
梯度消失: BPTT 把循环展开成深网络, 梯度连乘很多<1的项→指数衰减→早期步学不到→记不住长依赖
LSTM: cell state 信息高速公路(加性传递, 类比ResNet +x) + 三门(遗忘/输入/输出); 解决梯度消失
GRU: LSTM 简化版, 2个门(更新/重置), 参数更少更快, 效果常相当
实验: 加法问题 T=40, 朴素RNN退化到瞎猜, LSTM 仍能学 → 长依赖能力差距
RNN 局限→Transformer: ①无法并行(必须逐步) ②超长依赖仍吃力 → 注意力/Transformer(12.3)取代
```

### 💡 面试速查 / Interview cheat-sheet
1. **RNN**: 隐藏状态传递记忆, 时间上参数共享, 处理变长序列。
   RNN: hidden state carries memory, weight-sharing across time, handles variable length.
2. **梯度消失**: BPTT 连乘<1项指数衰减; 早期步学不到长依赖。
   Vanishing gradients: BPTT multiplies sub-1 terms → exponential decay; can't learn long deps.
3. **LSTM**: cell state 加性高速公路 + 遗忘/输入/输出门; 解决梯度消失。
   LSTM: additive cell-state highway + forget/input/output gates; fixes vanishing gradients.
4. **GRU**: 2门简化版, 更少参数更快, 常与LSTM相当。
   GRU: 2-gate simplification, fewer params/faster, often comparable.
5. **为何被Transformer取代**: RNN 必须串行(慢)、超长依赖仍难; 注意力可并行+直接建模任意距离。
   Why Transformers replaced RNN: serial (slow) + long deps still hard; attention parallel + any-distance.

### 下一节 / Next
**12.2 Seq2Seq 与注意力**——用 RNN 做"序列到序列"(如翻译): 编码器把输入压成一个向量、解码器生成输出。但"压成一个向量"是瓶颈。**注意力机制**让解码器每步直接"回看"输入的相关部分——这是通往 Transformer 的关键一步。
**12.2 Seq2Seq & Attention** — using RNNs for sequence-to-sequence (e.g. translation): an encoder compresses input into a vector, a decoder generates output. But "one vector" is a bottleneck. **Attention** lets the decoder "look back" at relevant input parts each step — the key step toward Transformers.
